# 🎨 Lab 07 · How machines invent: next-token prediction, flows, and flow matching

**World Models course · Lectures 7–10 · HW2** &nbsp;|&nbsp; ⏱ about 75 min &nbsp;|&nbsp; 💻 CPU is fine (training takes about a minute)

A world model must **imagine** futures, not only score them. That makes it a *generative model*. You'll build three families from scratch:

| Part | Family | Famous examples |
|---|---|---|
| A | **Autoregressive**: one token at a time | GPT, Gemini, Claude; Genie's latent-action video tokens |
| B | **Normalizing flow**: stretch a simple shape into a complex one | RealNVP, Apple TARFlow / STARFlow |
| C | **Flow matching**: learn to push noise into data along straight lines | Stable Diffusion 3, FLUX, Meta Movie Gen, and the action heads of **π0** and **NVIDIA GR00T** |

Part C is the foundation of the **capstone**, where the same ~30 lines generate robot actions instead of 2D points.

🧩 challenges · 🔮 predictions · 🎛️ playgrounds. Blank or wrong answers never break the notebook.

In [ ]:
#@title 🔧 Step 0 · Run this cell first (click ▶). It loads the tools for this lab. { display-mode: "form" }
import time, math
import numpy as np
import matplotlib.pyplot as plt
import torch, torch.nn as nn
from sklearn.datasets import make_moons
torch.manual_seed(0)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25})

# ---------------------------------------------------------------------------
# Guided-lab helpers. You never need to edit this cell.
#  * ___            : a blank for you to fill in
#  * check(name, x) : checks your answer; if it is blank or wrong, it explains
#                     and hands back a working version so the notebook keeps going
#  * quiz(id)       : a clickable multiple-choice question
#  * playground(...) : sliders that re-run a function when you let go
# ---------------------------------------------------------------------------
import inspect, html as _html
import numpy as np
from IPython.display import display, HTML
import os
try:
    import ipywidgets as widgets
    _WIDGETS = not os.environ.get("GUIDE_NO_WIDGETS")
except Exception:
    _WIDGETS = False

class BlankNotFilled(Exception):
    pass

class _Blank:
    """The ___ placeholder. Any maths with it stops with a friendly message."""
    __array_ufunc__ = None
    def _stop(self, *args, **kwargs):
        raise BlankNotFilled("There is still a ___ blank to fill in.")
    __add__ = __radd__ = __sub__ = __rsub__ = __mul__ = __rmul__ = _stop
    __truediv__ = __rtruediv__ = __floordiv__ = __rfloordiv__ = _stop
    __pow__ = __rpow__ = __matmul__ = __rmatmul__ = __mod__ = __rmod__ = _stop
    __neg__ = __pos__ = __abs__ = __getitem__ = __call__ = __iter__ = _stop
    __lt__ = __le__ = __gt__ = __ge__ = __bool__ = __float__ = __int__ = __index__ = _stop
    __array__ = __len__ = _stop
    def __getattr__(self, name):
        if name.startswith('__'):
            raise AttributeError(name)
        raise BlankNotFilled("There is still a ___ blank to fill in.")
    def __repr__(self):
        return "___"

___ = _Blank()
CHALLENGES, QUIZZES = {}, {}
_solved, _quiz_score = {}, {}

_STYLE = {
    "ok":   ("#e8f6ee", "#1b7a4b", "✅"),
    "wait": ("#fff5e0", "#9a5b00", "🧩"),
    "bad":  ("#fdecea", "#b3261e", "❌"),
    "info": ("#eaf1fb", "#245eb5", "💡"),
}

def card(kind, title, body=""):
    bg, fg, icon = _STYLE[kind]
    display(HTML(
        f'<div style="background:{bg};border-left:5px solid {fg};padding:10px 14px;'
        f'border-radius:6px;margin:6px 0;color:#1d2530;font-size:14px;line-height:1.5">'
        f'<b style="color:{fg}">{icon} {title}</b><div>{body}</div></div>'))

def _as_numpy(x):
    if hasattr(x, "detach"):
        x = x.detach().cpu().numpy()
    if isinstance(x, (list, tuple)):
        return [_as_numpy(v) for v in x]
    return x

def _same(a, b, tol):
    a, b = _as_numpy(a), _as_numpy(b)
    if isinstance(a, list) or isinstance(b, list):
        return isinstance(a, list) and isinstance(b, list) and len(a) == len(b) and all(_same(x, y, tol) for x, y in zip(a, b))
    try:
        a = np.asarray(a, dtype=float); b = np.asarray(b, dtype=float)
    except Exception:
        return a == b
    return a.shape == b.shape and np.allclose(a, b, atol=tol, rtol=tol)

def _has_blank(obj):
    if isinstance(obj, _Blank):
        return True
    if callable(obj):
        try:
            return "___" in inspect.getsource(obj)
        except Exception:
            return False
    return False

def check(name, answer):
    """Check a challenge. Returns your answer if it works, otherwise a working reference."""
    ch = CHALLENGES[name]
    ref = ch["reference"]
    title = ch.get("title", name)
    fallback = ("<br><i>For now the notebook will use a working version so every later cell still runs. "
                "Come back, fill it in, and re-run this cell.</i>")
    if _has_blank(answer):
        _solved.setdefault(name, False)
        card("wait", f"Challenge “{title}” is waiting for you", "Hint: " + ch["hint"] + fallback)
        return ref
    try:
        if "test" in ch:
            ok, message = ch["test"](answer)
        elif callable(ref):
            ok, message = True, ""
            for args in ch["cases"]:
                args = args if isinstance(args, tuple) else (args,)
                expected, got = ref(*args), answer(*args)
                if not _same(expected, got, ch.get("tol", 1e-6)):
                    ok = False
                    message = "For a test input your function gave a different result from the expected one."
                    break
        else:
            ok = _same(ref, answer, ch.get("tol", 1e-6))
            message = f"You entered <code>{_html.escape(repr(_as_numpy(answer)))}</code>."
    except BlankNotFilled:
        _solved.setdefault(name, False)
        card("wait", f"Challenge “{title}” still has a blank", "Hint: " + ch["hint"] + fallback)
        return ref
    except Exception as err:
        ok, message = False, f"Running your version raised <code>{_html.escape(type(err).__name__)}: {_html.escape(str(err))}</code>."
    if ok:
        _solved[name] = True
        card("ok", f"Challenge solved: {title}", ch.get("why", ""))
        return answer
    _solved[name] = False
    card("bad", f"Not quite yet: {title}", message + "<br>Hint: " + ch["hint"] + fallback)
    return ref

def quiz(qid):
    q = QUIZZES[qid]
    question = f'<div style="font-size:15px;margin:8px 0 4px"><b>{"🔮 Predict: " if q.get("predict") else "🤔 "}{q["q"]}</b></div>'
    if not _WIDGETS:
        options = "".join(f"<li>{_html.escape(o)}</li>" for o in q["options"])
        display(HTML(question + f"<ol type='A'>{options}</ol><details><summary>Answer</summary>"
                     f"{'ABCDEFG'[q['answer']]}. {q['explain']}</details>"))
        return
    out = widgets.Output()
    buttons = []
    def choose(i):
        def handler(_):
            _quiz_score.setdefault(qid, i == q["answer"])
            for j, b in enumerate(buttons):
                b.button_style = "success" if j == q["answer"] else ("danger" if j == i else "")
            with out:
                out.clear_output()
                if i == q["answer"]:
                    card("ok", "Yes!", q["explain"])
                else:
                    card("bad", "Not this one. Here is the reasoning:", q["explain"])
        return handler
    for i, option in enumerate(q["options"]):
        b = widgets.Button(description=f"{'ABCDEFG'[i]}. {option}", layout=widgets.Layout(width="auto", max_width="100%"))
        b.on_click(choose(i))
        buttons.append(b)
    display(HTML(question), widgets.VBox(buttons), out)

def playground(fn, **controls):
    """controls: name=(min, max, step, default) for sliders, or name=[option, ...] for a dropdown."""
    defaults, sliders = {}, {}
    for name, spec in controls.items():
        if isinstance(spec, list):
            defaults[name] = spec[0]
            if _WIDGETS:
                sliders[name] = widgets.Dropdown(options=spec, value=spec[0], description=name)
        else:
            lo, hi, step, value = spec
            defaults[name] = value
            if _WIDGETS:
                kind = widgets.IntSlider if all(isinstance(v, int) for v in spec) else widgets.FloatSlider
                sliders[name] = kind(min=lo, max=hi, step=step, value=value, description=name,
                                     continuous_update=False, style={"description_width": "initial"},
                                     layout=widgets.Layout(width="420px"))
    if _WIDGETS:
        ui = widgets.VBox(list(sliders.values()))
        out = widgets.interactive_output(fn, sliders)
        display(ui, out)
    else:
        fn(**defaults)

def progress_report():
    solved = sum(_solved.values()); total = len(CHALLENGES)
    right = sum(_quiz_score.values()); asked = len(_quiz_score)
    stars = "⭐" * solved + "☆" * (total - solved)
    body = f"Challenges solved yourself: <b>{solved} / {total}</b> {stars}<br>"
    body += f"Quiz questions right on the first click: <b>{right} / {asked}</b> (of {len(QUIZZES)} in this lab)"
    missing = [CHALLENGES[k].get('title', k) for k in CHALLENGES if not _solved.get(k)]
    if missing:
        body += "<br>Still worth a try: " + ", ".join(missing)
    card("info", "Your progress in this lab", body)

# ---- this lab's challenges and quizzes ----
CHALLENGES["normalise"] = dict(title="Counts → probabilities",
    reference=lambda counts: counts / counts.sum(axis=-1, keepdims=True),
    cases=[np.array([[1., 3.], [2., 2.]]), np.array([[[1., 1., 2.]]])],
    hint="Divide each row of counts by that row's total. Use <code>axis=-1, keepdims=True</code> so the shapes line up.",
    why="Probabilities of all possible next tokens must add up to 1. This is what the <b>softmax</b> layer of an LLM guarantees.")

def _ref_temp(probs, temperature):
    p = probs ** (1.0 / temperature)
    return p / p.sum()
CHALLENGES["temperature"] = dict(title="Temperature", reference=_ref_temp,
    cases=[(np.array([0.7, 0.2, 0.1]), 0.5), (np.array([0.7, 0.2, 0.1]), 2.0), (np.array([0.25, 0.75]), 1.0)],
    hint="Raise every probability to the power 1/temperature, then renormalise so they add up to 1.",
    why="Low temperature sharpens toward the most likely token (safe, repetitive); high temperature flattens (creative, error-prone). It's the same knob you set in LLM APIs.")

def _ref_flow_logp(y, scale, shift):
    x = (y - shift) / scale
    return -0.5 * x**2 - 0.5 * np.log(2 * np.pi) - np.log(abs(scale))
CHALLENGES["change_of_variables"] = dict(title="Stretching changes density", reference=_ref_flow_logp,
    cases=[(np.array([0.0, 3.0, 5.5]), 2.0, 3.0), (np.array([1.0]), 0.5, -1.0)],
    hint="Undo the flow to get x, take the Gaussian log-density of x, then subtract <code>np.log(abs(scale))</code>: stretching by 2 spreads probability twice as thin.",
    why="This is the <b>change-of-variables formula</b>. Flows can compute exact likelihoods because every layer's stretch (its Jacobian determinant) is easy to compute.")

CHALLENGES["interpolate"] = dict(title="A straight path from noise to data",
    reference=lambda x0, x1, t: (1 - t) * x0 + t * x1,
    cases=[(torch.tensor([[0., 0.]]), torch.tensor([[2., 4.]]), torch.tensor([[0.25]])), (torch.randn(5, 2), torch.randn(5, 2), torch.rand(5, 1))],
    hint="At t=0 you should get the noise x0; at t=1 the data x1; in between, a weighted mix: (1 − t)·x0 + t·x1.",
    why="Flow matching trains on points along <b>straight lines</b> between noise and data. This is the “rectified flow” path used by Stable Diffusion 3 and π0.")

CHALLENGES["target_velocity"] = dict(title="Which way should the point move?",
    reference=lambda x0, x1: x1 - x0,
    cases=[(torch.tensor([[0., 0.]]), torch.tensor([[2., 4.]])), (torch.randn(4, 2), torch.randn(4, 2))],
    hint="If you travel from x0 to x1 in one unit of time at constant speed, your velocity is the displacement.",
    why="Along a straight line the velocity is constant: <code>x1 − x0</code>. The network only has to learn this simple regression target, with no adversary (GAN) and no noise schedule maths (DDPM).")

def _ref_euler(x, velocity_fn, n_steps):
    dt = 1.0 / n_steps
    for i in range(n_steps):
        t = torch.full((len(x), 1), i * dt)
        x = x + dt * velocity_fn(x, t)
    return x
def _test_euler(fn):
    x = torch.zeros(3, 2)
    const = lambda x, t: torch.ones_like(x) * torch.tensor([1.0, -2.0])
    grow = lambda x, t: x + t
    ok = torch.allclose(fn(x, const, 4), torch.tensor([[1.0, -2.0]] * 3)) and torch.allclose(fn(torch.ones(2, 2), grow, 5), _ref_euler(torch.ones(2, 2), grow, 5))
    return ok, "Each step should move x by dt × velocity at the current (x, t)."
CHALLENGES["euler"] = dict(title="Generate by following the velocity", reference=_ref_euler, test=_test_euler,
    hint="Take a small step in the direction of the predicted velocity: x + dt × velocity_fn(x, t).",
    why="Sampling is solving an ODE with <b>Euler steps</b>, just like lab 01's puck. Fewer steps means faster generation. That's why flow matching is popular for real-time robot control.")

QUIZZES["context"] = dict(predict=True, q="A model sees only the <b>last 1</b> position on the ring. When it generates, what will its path look like?",
    options=["A smooth circle in one direction", "Jittery back-and-forth: it can't tell which direction the episode is going", "It will stop moving"],
    answer=1, explain="From one position, clockwise and counter-clockwise are equally likely, so each sampled step is a coin flip. Two tokens of context reveal the direction. This is state aliasing (lab 02) in a language model.")
QUIZZES["one_step"] = dict(predict=True, q="If we generate with just <b>1</b> Euler step, the samples will look like…",
    options=["perfect moons", "a blurry blob near the middle of the data", "pure noise"],
    answer=1, explain="One step follows the <i>average</i> direction from each noise point. Averages of many possible targets land between the moons. More steps let paths bend to specific destinations.")
QUIZZES["families"] = dict(q="Which family gives an <b>exact</b> log-likelihood with a single forward pass through invertible layers?",
    options=["GAN", "VAE", "Normalizing flow", "Diffusion / flow matching sampled with a few Euler steps"],
    answer=2, explain="Flows are invertible with tractable Jacobians, so the change-of-variables formula gives exact likelihood. VAEs give a lower bound (ELBO), GANs give none, and diffusion/flow matching need an extra ODE integration to estimate it.")
print('✅ Setup complete. Scroll down and run the cells in order.')

---
# Part A · One token at a time 🔤

## 1 · A tiny language
A ball moves around a **ring of 12 positions**. Each position is a *token* (0–11). In each episode the ball goes clockwise **or** counter-clockwise, always one step at a time.

In [ ]:
VOCAB, LENGTH = 12, 25
rng = np.random.default_rng(6280)
episodes = []
for _ in range(240):
    start, direction = rng.integers(VOCAB), rng.choice([-1, 1])
    episodes.append((start + direction * np.arange(LENGTH)) % VOCAB)
episodes = np.array(episodes)
train, test = episodes[:180], episodes[180:]
print("two example 'sentences':\n", train[0], "\n", train[1])

## 2 · Learn P(next token | recent tokens) by counting
With `context = 2`, the model keeps a table: *"after tokens (a, b), how often did each next token appear?"* Then it turns counts into probabilities.

### 🧩 Challenge 1 · Counts → probabilities

In [ ]:
def normalise(counts):
    return ___     # 🧩 each row must add up to 1

normalise = check("normalise", normalise)

def fit(data, context):
    counts = np.full((VOCAB,) * (context + 1), 0.01)          # tiny pseudo-count avoids log(0)
    for seq in data:
        for t in range(context, len(seq)):
            counts[tuple(seq[t - context:t + 1])] += 1
    return normalise(counts)

def avg_neg_log_likelihood(table, data, context):
    return np.mean([-np.log(table[tuple(seq[t - context:t + 1])]) for seq in data for t in range(2, len(seq))])

tables = {c: fit(train, c) for c in (1, 2)}
for c, table in tables.items():
    print(f"context {c}: test loss (negative log-likelihood) = {avg_neg_log_likelihood(table, test, c):.3f}")
print(f"(a coin flip between 2 options costs log 2 = {np.log(2):.3f})")

<details><summary>🤔 <b>Need a hint?</b></summary>

Divide by the sum over the last axis; keep that axis so broadcasting works.

</details>
<details><summary>🔑 <b>Show the answer</b> (try first!)</summary>

<pre>return counts / counts.sum(axis=-1, keepdims=True)     # 🧩 each row must add up to 1</pre>

</details>

In [ ]:
quiz("context")

## 3 · Generate, with a temperature knob 🌡️
To generate, repeatedly sample the next token from the model's probabilities and append it.

### 🧩 Challenge 2 · Temperature

In [ ]:
def apply_temperature(probs, temperature):
    p = ___     # 🧩 sharpen (T<1) or flatten (T>1)
    return p / p.sum()

apply_temperature = check("temperature", apply_temperature)

def generate(table, context, prefix, n, temperature, seed):
    r = np.random.default_rng(seed)
    seq = list(prefix)
    while len(seq) < n:
        probs = apply_temperature(table[tuple(seq[-context:])], temperature)
        seq.append(int(r.choice(VOCAB, p=probs)))
    return np.array(seq)

<details><summary>🤔 <b>Need a hint?</b></summary>

Raise to the power <code>1 / temperature</code>. The next line renormalises for you.

</details>
<details><summary>🔑 <b>Show the answer</b> (try first!)</summary>

<pre>p = probs ** (1.0 / temperature)     # 🧩 sharpen (T&lt;1) or flatten (T&gt;1)</pre>

</details>

### 🎛️ Playground · Context length and temperature
The prefix comes from a real test episode. Watch for direction changes.

In [ ]:
def ar_lab(context=1, temperature=1.0, seed=0):
    real = test[3]
    fig, ax = plt.subplots(figsize=(8, 3))
    ax.plot(real, "ko-", alpha=0.3, label="a real episode")
    changes = []
    for s in range(100):
        g = generate(tables[context], context, real[:2], LENGTH, temperature, seed * 1000 + s)
        steps = (np.diff(g) + VOCAB // 2) % VOCAB - VOCAB // 2
        changes.append(np.mean(steps[1:] != steps[:-1]))
        if s < 3: ax.plot(g, ".--", label=f"sample {s + 1}")
    ax.set(xlabel="time", ylabel="token (position on ring)", title=f"context={context}, T={temperature}: direction changes in {np.mean(changes):.0%} of steps (100 samples)")
    ax.legend(fontsize=7, ncol=4); plt.show()

playground(ar_lab, context=[1, 2], temperature=(0.2, 3.0, 0.1, 1.0), seed=(0, 20, 1, 0))

**Teacher forcing vs. free running:** training loss uses the *real* previous tokens. Generation feeds the model's *own* samples back in, so one bad sample can derail everything after it (lab 01's compounding error again). That's why a good test loss doesn't guarantee good long generations.

---
# Part B · Normalizing flows: stretch a simple shape 🫧

Start with a bell curve `x ~ N(0, 1)`. Apply an invertible map `y = scale · x + shift`. Where do the samples go, and what is the new **density**?

If you stretch by 2, the same probability is spread over twice the width, so the density **halves**:

$$\log p_Y(y) = \log p_X\!\left(\frac{y - \text{shift}}{\text{scale}}\right) - \log|\text{scale}|$$

### 🧩 Challenge 3 · Stretching changes density

In [ ]:
def flow_log_density(y, scale, shift):
    x = (y - shift) / scale                                   # undo the flow
    log_px = -0.5 * x**2 - 0.5 * np.log(2 * np.pi)            # Gaussian log-density of x
    return ___                      # 🧩 correct for the stretch

flow_log_density = check("change_of_variables", flow_log_density)

<details><summary>🤔 <b>Need a hint?</b></summary>

Subtract the log of how much you stretched.

</details>
<details><summary>🔑 <b>Show the answer</b> (try first!)</summary>

<pre>return log_px - np.log(abs(scale))                      # 🧩 correct for the stretch</pre>

</details>

In [ ]:
def flow_lab(scale=2.0, shift=3.0):
    y = scale * np.random.default_rng(0).normal(size=20000) + shift
    grid = np.linspace(-8, 12, 400)
    plt.figure(figsize=(6.5, 2.8))
    plt.hist(y, bins=120, density=True, alpha=0.4, label="samples pushed through the flow")
    plt.plot(grid, np.exp(flow_log_density(grid, scale, shift)), lw=2, label="exact density formula")
    plt.xlim(-8, 12); plt.legend(fontsize=8); plt.title(f"y = {scale}·x + {shift}"); plt.show()

playground(flow_lab, scale=(0.3, 4.0, 0.1, 2.0), shift=(-3.0, 6.0, 0.5, 3.0))

Real flows (RealNVP, Glow, TARFlow) stack many such invertible layers, each stretching *different* coordinates depending on the others (**coupling layers**), and add up each layer's `log|stretch|`. Exact likelihood is their superpower. The price is that every layer must stay invertible.

In [ ]:
quiz("families")

---
# Part C · Flow matching from scratch 🌊

**Goal:** turn random noise into points shaped like two interlocking moons.

**The idea in one picture:** pair each noise point `x0` with a data point `x1` and draw a straight line between them. Train a network to predict **which way to move** at any point `x_t` along any line at any time `t`. To generate, start from fresh noise and follow the predicted directions.

In [ ]:
def sample_data(n):
    x, _ = make_moons(n, noise=0.05)
    return torch.tensor((x - [0.5, 0.25]) * 1.6, dtype=torch.float32)   # centred, roughly unit scale

data = sample_data(3000)
plt.figure(figsize=(4, 3)); plt.scatter(*data.T, s=2); plt.title("the data we want to generate"); plt.axis("equal"); plt.show()

### 🧩 Challenge 4 · A straight path from noise to data

In [ ]:
def interpolate(x0, x1, t):
    return ___     # 🧩 t=0 → noise, t=1 → data

interpolate = check("interpolate", interpolate)

<details><summary>🤔 <b>Need a hint?</b></summary>

A weighted average whose weights depend on t.

</details>
<details><summary>🔑 <b>Show the answer</b> (try first!)</summary>

<pre>return (1 - t) * x0 + t * x1     # 🧩 t=0 → noise, t=1 → data</pre>

</details>

### 🧩 Challenge 5 · Which way should the point move?

In [ ]:
def target_velocity(x0, x1):
    return ___                   # 🧩 constant velocity along the straight line

target_velocity = check("target_velocity", target_velocity)

<details><summary>🤔 <b>Need a hint?</b></summary>

Displacement per unit time.

</details>
<details><summary>🔑 <b>Show the answer</b> (try first!)</summary>

<pre>return x1 - x0                   # 🧩 constant velocity along the straight line</pre>

</details>

### 🎛️ Playground · Slide along the paths
Each grey line joins one noise point to one data point. The dots show where every point is at time `t`.

In [ ]:
x0_demo, x1_demo = torch.randn(400, 2), sample_data(400)
def paths(t=0.5):
    xt = interpolate(x0_demo, x1_demo, torch.full((400, 1), t))
    plt.figure(figsize=(4.5, 4))
    for i in range(60):
        plt.plot([x0_demo[i, 0], x1_demo[i, 0]], [x0_demo[i, 1], x1_demo[i, 1]], c="gray", lw=0.4, alpha=0.5)
    plt.scatter(*xt.T, s=5, c="tab:purple"); plt.xlim(-3, 3); plt.ylim(-3, 3); plt.title(f"t = {t:.2f}"); plt.show()

playground(paths, t=(0.0, 1.0, 0.05, 0.5))

## 4 · Train the velocity network 🏋️
A small neural network takes `(x_t, t)` and outputs a 2D velocity. The loss is plain **mean squared error** against `target_velocity`. Read the loop line by line; it's the whole algorithm.

In [ ]:
class VelocityNet(nn.Module):
    def __init__(self, hidden=128, n_labels=0):
        super().__init__()
        self.label_emb = nn.Embedding(n_labels, 16) if n_labels else None
        extra = 16 if n_labels else 0
        self.net = nn.Sequential(
            nn.Linear(2 + 8 + extra, hidden), nn.SiLU(),
            nn.Linear(hidden, hidden), nn.SiLU(),
            nn.Linear(hidden, hidden), nn.SiLU(),
            nn.Linear(hidden, 2))
    def forward(self, x, t, label=None):
        freqs = torch.arange(1, 5, dtype=torch.float32) * math.pi            # describe t with a few waves
        t_feat = torch.cat([torch.sin(freqs * t), torch.cos(freqs * t)], dim=1)
        parts = [x, t_feat] + ([self.label_emb(label)] if label is not None else [])
        return self.net(torch.cat(parts, dim=1))

def train_flow_matching(model, get_batch, steps=3000, batch=512, lr=2e-3):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    losses = []
    for step in range(steps):
        x1, label = get_batch(batch)                     # real data (and optional label)
        x0 = torch.randn_like(x1)                        # fresh noise
        t = torch.rand(len(x1), 1)                       # a random time for each pair
        xt = interpolate(x0, x1, t)                      # a point somewhere on the straight line
        loss = ((model(xt, t, label) - target_velocity(x0, x1)) ** 2).mean()
        opt.zero_grad(); loss.backward(); opt.step()     # nudge the weights to reduce the error
        losses.append(loss.item())
    return losses

torch.manual_seed(0)
model = VelocityNet()
t0 = time.time()
losses = train_flow_matching(model, lambda n: (sample_data(n), None))
print(f"trained in {time.time() - t0:.0f} s")
plt.figure(figsize=(5, 2.5)); plt.plot(np.convolve(losses, np.ones(50) / 50, "valid")); plt.xlabel("step"); plt.ylabel("loss"); plt.title("flow matching loss"); plt.show()

Why doesn't the loss go to zero? Many different lines pass through the same point `x_t`, pointing to different data points. The network can only learn their **average** direction, and that average is exactly what makes generation work.

## 5 · Generate! ✨
### 🧩 Challenge 6 · Generate by following the velocity

In [ ]:
def euler_sample(x, velocity_fn, n_steps):
    dt = 1.0 / n_steps
    for i in range(n_steps):
        t = torch.full((len(x), 1), i * dt)
        x = ___     # 🧩 take a small step along the predicted velocity
    return x

euler_sample = check("euler", euler_sample)

<details><summary>🤔 <b>Need a hint?</b></summary>

New position = position + dt × velocity. Same as lab 01's puck.

</details>
<details><summary>🔑 <b>Show the answer</b> (try first!)</summary>

<pre>x = x + dt * velocity_fn(x, t)     # 🧩 take a small step along the predicted velocity</pre>

</details>

In [ ]:
quiz("one_step")

In [ ]:
real = sample_data(2000)
def quality(samples):
    d = torch.cdist(samples, real)                   # distance from every sample to every real point
    precision = d.min(dim=1).values.mean().item()    # are samples close to SOME real point?
    coverage = (d.min(dim=0).values < 0.15).float().mean().item()   # is every part of the data reached?
    return precision, coverage

def generate_lab(n_steps=10):
    torch.manual_seed(1)
    with torch.no_grad():
        t0 = time.perf_counter()
        samples = euler_sample(torch.randn(2000, 2), lambda x, t: model(x, t), n_steps)
        ms = 1000 * (time.perf_counter() - t0)
    prec, cov = quality(samples)
    plt.figure(figsize=(4.5, 3.5)); plt.scatter(*real.T, s=1, c="lightgray", label="real")
    plt.scatter(*samples.T, s=2, c="tab:purple", label="generated"); plt.xlim(-3, 3); plt.ylim(-2.5, 2.5)
    plt.title(f"{n_steps} step(s) · {ms:.0f} ms"); plt.legend(fontsize=7, loc="lower left"); plt.show()
    print(f"avg distance to nearest real point: {prec:.3f} (lower is better) · data covered: {cov:.0%}")

playground(generate_lab, n_steps=(1, 50, 1, 10))

In [ ]:
print("steps | distance to data | coverage")
for n in [1, 2, 5, 10, 50]:
    with torch.no_grad():
        torch.manual_seed(1)
        s = euler_sample(torch.randn(2000, 2), lambda x, t: model(x, t), n)
    p, c = quality(s)
    print(f"{n:5d} | {p:16.3f} | {c:7.0%}")

## 6 · ✨ Tell the generator what you want: conditioning
Give the network an extra input, a **label** (0 = top moon, 1 = bottom moon), and it learns to generate *that* moon on request.

🤖 **This is the capstone idea:** replace *2D points* with **a chunk of 16 future robot actions**, and replace the *label* with **what the robot sees** (camera features + instruction). You get a **flow-matching action expert**, the component that makes π0 and GR00T move.

In [ ]:
def labelled_batch(n):
    x, y = make_moons(n, noise=0.05)
    return torch.tensor((x - [0.5, 0.25]) * 1.6, dtype=torch.float32), torch.tensor(y)

torch.manual_seed(0)
cond_model = VelocityNet(n_labels=2)
t0 = time.time(); train_flow_matching(cond_model, labelled_batch); print(f"trained in {time.time() - t0:.0f} s")

fig, axs = plt.subplots(1, 2, figsize=(8, 3.3))
for label, ax in zip([0, 1], axs):
    lab = torch.full((1000,), label)
    with torch.no_grad():
        s = euler_sample(torch.randn(1000, 2), lambda x, t: cond_model(x, t, lab), 20)
    ax.scatter(*real.T, s=1, c="lightgray"); ax.scatter(*s.T, s=2, c=["tab:blue", "tab:orange"][label])
    ax.set(xlim=(-3, 3), ylim=(-2.5, 2.5), title=f"generate label {label}")
plt.tight_layout(); plt.show()

---
## 7 · Recap: the generative family tree (HW2 Part D) 🌳

| Family | Trains by | Generates by | Likelihood? | Classic failure |
|---|---|---|---|---|
| VAE | reconstruction + KL (ELBO) | decode a sampled latent | lower bound | blurry samples, posterior collapse |
| GAN | generator vs discriminator game | one forward pass | none | mode collapse, unstable training |
| Autoregressive | next-token log-likelihood | sample token by token | exact | compounding errors, slow for long outputs |
| Normalizing flow | exact log-likelihood | one invertible pass | exact | invertibility limits architecture |
| DDPM (diffusion) | predict added noise | many denoising steps | bound / ODE estimate | slow sampling |
| **Flow matching** | regress straight-line velocity | a few ODE steps | ODE estimate | too few steps → blur |

⚠️ *Generation time* `t` (noise → data) is **not** physical time in the world. A video world model has both: flow time inside each frame's generation, and real time between frames.

### 🏭 Where these run today
* **Flow matching:** Stable Diffusion 3 & FLUX images, Meta **Movie Gen** video, **π0 / π0.5** (Physical Intelligence) and **GR00T N1** (NVIDIA) robot action heads, **SmolVLA** (Hugging Face).
* **Diffusion world models:** DIAMOND (Atari), GameNGen (Doom), NVIDIA **Cosmos**, DeepMind **Genie 3** and **Veo**.
* **Autoregressive:** every LLM; Genie's latent-action video model.

### 🧪 HW2, beginner version
1. Train with `steps=500` instead of 3000. Which degrades first: distance to data or coverage?
2. In the generate playground, find the smallest number of steps whose samples you'd call "good". How many milliseconds does it take?
3. **Optional deeper dive:** MIT 6.S184 Lab 2 (linked in `GENERATION_ASSIGNMENT.md`) derives flow matching and score matching formally.

### 🗣️ Explain it back
Explain flow matching to a friend in three sentences, using the words *noise*, *straight line* and *velocity*.

In [ ]:
progress_report()